# M3 - Random Forest avec Target Encoding et Optuna

**Objectif** : Entraîner un modèle Random Forest optimisé sur le dataset Ames Housing pour prédire le prix de vente des maisons.

**Différences clés par rapport à la baseline M0** :
- `LinearRegression` → `RandomForestRegressor` : passage d'un modèle linéaire (qui suppose une relation proportionnelle entre les variables et le prix) à un modèle à base d'arbres de décision, capable de capturer des relations non-linéaires et des interactions complexes entre variables.
- **5 variables** → **toutes les variables** : le M0 se limite à 5 features sélectionnées manuellement, le M3 utilise l'ensemble des colonnes du dataset puis laisse un filtre Lasso (`SelectFromModel`) éliminer automatiquement les variables inutiles.
- **Feature engineering étendu** : ajout de variables combinées (`TotalSF`, `TotalBath`, `IsRemodeled`) et d'un encodage ordinal des variables de qualité (`ExterQual`, `KitchenQual`, etc.), absents du M0.
- **Optimisation Optuna** : le Random Forest a de nombreux hyperparamètres à régler (`n_estimators`, `max_depth`, etc.). On utilise Optuna (recherche bayésienne avec pruning et early stopping) pour les optimiser automatiquement, ce qui n'était pas nécessaire dans le M0 (la régression linéaire n'a pas d'hyperparamètres).

In [6]:
# M3 - Random Forest + Target Encoding + Optuna Tuning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing classique
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from category_encoders import TargetEncoder
from sklearn.impute import KNNImputer

# Outils pour la sélection de variables et l'optimisation
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.exceptions import TrialPruned


# 2. Chargement des Données et Feature Engineering
- On charge `train.csv` et `test.csv`, puis on applique un feature engineering avancé :
    - **Variables combinées** : `TotalSF` (surface totale), `TotalBath` (salles de bain totales), `IsRemodeled` (flag de rénovation).
    - **Encodage ordinal** : Les variables de qualité (`ExterQual`, `KitchenQual`, etc.) sont converties en chiffres de 0 à 5 pour créer une hiérarchie stricte.
    - **Nettoyage des années** : suppression des colonnes brutes (`YearBuilt`, `YearRemodAdd`, `GarageYrBlt`) car elles sont remplacées par `AgeBuilt` et `AgeRemod`.
- Suppression des outliers critiques (grandes maisons à prix anormalement bas).
- La cible `SalePrice` est transformée en `log1p` pour réduire l'asymétrie de sa distribution.
- Séparation en `X_train` / `y_train` / `X_test`.

In [7]:
def advanced_feature_engineering(data):
    """
    Applique les transformations métiers sur le jeu de données.
    Note : L'encodage ordinal et les variables combinées (TotalSF, TotalBath, IsRemodeled)
    sont inspirés de stratégies performantes trouvées sur les notebooks publics de Kaggle.
    """
    df = data.copy()

    # --- A. CRÉATION DE VARIABLES (Inspiration Kaggle) ---
    # 1. Surface totale (Sous-sol + RDC + 1er étage)
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

    # 2. Salles de bain totales (Les "HalfBath" comptent pour 0.5)
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']

    # 3. Rénovation (Flag binaire : 1 si rénové, 0 sinon)
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)

    # --- B. ENCODAGE ORDINAL (Inspiration Kaggle) ---
    # On force la qualité en chiffres pour créer une hiérarchie stricte pour le Random Forest
    ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                    'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
    quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0, np.nan: 0}

    for col in ordinal_cols:
        if col in df.columns:
            df[col] = df[col].map(quality_map)

    # --- C. GESTION DES DATES (Calcul des âges) ---
    df['AgeBuilt'] = df['YrSold'] - df['YearBuilt']
    df['AgeRemodAdd'] = df['YrSold'] - df['YearRemodAdd']
    df['AgeGarage'] = df['YrSold'] - df['GarageYrBlt']

    # On supprime les anciennes dates brutes qui perturbent le modèle
    df = df.drop(['YearBuilt', 'YearRemodAdd', 'GarageYrBlt'], axis=1)

    return df


# --- 1. Chargement brut ---
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# --- 2. Nettoyage des Outliers (SUR LE TRAIN UNIQUEMENT) ---
# Suppression des 2 transactions aberrantes repérées visuellement (>4000 sqft & <300000$)
train_df = train_df.drop(train_df[train_df['GrLivArea'] > 4000].index)

# --- 3. Feature Engineering ---
train_df = advanced_feature_engineering(train_df)
test_df = advanced_feature_engineering(test_df)

# --- 4. Séparation X et y ---
X_train = train_df.drop(['Id', 'SalePrice'], axis=1)
y_train = pd.Series(np.log1p(train_df['SalePrice']), index=train_df.index)

X_test = test_df.drop(['Id'], axis=1)
test_ids = test_df['Id']

# 3. Création du Preprocessor
- On sépare les colonnes en 3 groupes, chacun traité par un pipeline adapté :
    1. **Numériques** : `KNNImputer` (imputation intelligente par les 5 voisins les plus proches) puis `RobustScaler` (mise à l'échelle résistante aux outliers).
    2. **Catégorielles « faux manquants »** (ex: `PoolQC`, `GarageType`) : les NaN signifient « il n'y en a pas ». On les remplace par `"None"` avant le `TargetEncoder`.
    3. **Catégorielles « vrais manquants »** (ex: `MSZoning`, `Electrical`) : les NaN sont de vraies données manquantes. On les remplace par la valeur la plus fréquente avant le `TargetEncoder`.
- Le `TargetEncoder` remplace chaque catégorie par la moyenne lissée du prix cible (avec `smoothing=10` pour éviter les fuites de données sur les catégories rares).

In [8]:
# Redétection dynamique des colonnes après les changements de l'étape 2
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# 1. Identification des "faux manquants" (les variables où NaN = "Il n'y en a pas")
cat_none_features = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                     'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                     'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                     'BsmtFinType2', 'MasVnrType']

# Les autres catégorielles (les "vrais manquants")
cat_freq_features = [col for col in categorical_features if col not in cat_none_features]

# 2. Pipeline Numérique : Utilisation du KNNImputer pour estimer les valeurs manquantes intelligemment
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', RobustScaler())
])

# 3. Pipeline Catégorielle 1 : Remplacement par "None" pour les absences de structure
cat_none_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('target_encoder', TargetEncoder(smoothing=10, handle_unknown='value', handle_missing='value'))
])

# 4. Pipeline Catégorielle 2 : Remplacement classique par la valeur la plus fréquente
cat_freq_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder(smoothing=10, handle_unknown='value', handle_missing='value'))
])

# 5. L'assembleur final (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat_none', cat_none_transformer, cat_none_features),
        ('cat_freq', cat_freq_transformer, cat_freq_features)
    ])

# 4. Diagnostic du Preprocessing (Target Encoding)
- Avant de lancer l'entraînement, on vérifie que le `TargetEncoder` va bien traiter toutes les variables catégorielles.
- On affiche la cardinalité de chaque variable (nombre de catégories distinctes) : une variable avec 25 catégories comme `Neighborhood` serait explosive en OneHotEncoding (25 colonnes), mais compacte en Target Encoding (1 seule colonne).
- On vérifie aussi la valeur la plus fréquente de chaque catégorie pour détecter d'éventuelles anomalies.

In [9]:
print("=" * 60)
print("DIAGNOSTIC DU PREPROCESSING (Target Encoding)")
print("=" * 60)
print(f"\n📊 Variables numériques : {len(numeric_features)}")
print(f"   {list(numeric_features)[:5]}... (affichage tronqué)\n")

print(f"📂 Variables catégorielles : {len(categorical_features)}")
for col in categorical_features:
    card = X_train[col].nunique()
    print(f"   {col:20} → cardinalité: {card:3} (top: {X_train[col].value_counts().index[0]})")

print(f"\n✔️  Target Encoding va traduire chaque catégorie en score numérique")
print(f"   (moyenne conditionnelle de SalePrice Log avec régularisation smoothing=10)")
print("=" * 60)

DIAGNOSTIC DU PREPROCESSING (Target Encoding)

📊 Variables numériques : 48
   ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond']... (affichage tronqué)

📂 Variables catégorielles : 34
   MSZoning             → cardinalité:   5 (top: RL)
   Street               → cardinalité:   2 (top: Pave)
   Alley                → cardinalité:   2 (top: Grvl)
   LotShape             → cardinalité:   4 (top: Reg)
   LandContour          → cardinalité:   4 (top: Lvl)
   Utilities            → cardinalité:   2 (top: AllPub)
   LotConfig            → cardinalité:   5 (top: Inside)
   LandSlope            → cardinalité:   3 (top: Gtl)
   Neighborhood         → cardinalité:  25 (top: NAmes)
   Condition1           → cardinalité:   9 (top: Norm)
   Condition2           → cardinalité:   8 (top: Norm)
   BldgType             → cardinalité:   5 (top: 1Fam)
   HouseStyle           → cardinalité:   8 (top: 1Story)
   RoofStyle            → cardinalité:   6 (top: Gable)
   RoofMatl            

# 5. Construction de la Pipeline Hybride et Tuning Optuna
- La pipeline M3 est un assemblage en 3 étapes :
    1. **Preprocessor** : le `ColumnTransformer` défini à l'étape 3 (KNNImputer + TargetEncoder).
    2. **Filtre Lasso** (`SelectFromModel` + `LassoCV`) : une régression Lasso qui élimine automatiquement les variables inutiles en mettant leurs coefficients à zéro. C'est un pré-filtre qui réduit le bruit avant d'alimenter le Random Forest.
    3. **Random Forest** : le modèle final qui apprend des relations non-linéaires complexes entre les variables restantes et le prix.
- L'optimisation des hyperparamètres est faite avec **Optuna** (recherche bayésienne), qui est plus efficace qu'un Grid Search classique : il apprend des essais précédents pour explorer en priorité les zones prometteuses de l'espace des hyperparamètres.
- Un mécanisme d'**early stopping** arrête la recherche si aucune amélioration n'est observée après 10 essais consécutifs.

In [ ]:
# Étape A : Le Filtre Lasso
# SelectFromModel va utiliser un LassoCV en interne. Il gardera uniquement les variables dont le coefficient n'a pas été réduit à 0.
feature_selector = SelectFromModel(LassoCV(cv=5, random_state=42))

# Étape B : Le Modèle Random Forest (Artillerie lourde)
# On initialise une Forêt Aléatoire de base
rf_model = RandomForestRegressor(random_state=42)

# Étape C : Assemblage de la Super Pipeline
pipeline_m3 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', feature_selector),
    ('rf', rf_model)
])

# Étape D : Optimisation avec Optuna (plus intelligent que RandomizedSearchCV)
def objective(trial):
    """
    Fonction objective pour Optuna : teste différentes combos d'hyperparamètres
    et retourne le RMSE moyen en validation croisée.

    Implémentation avec boucle CV manuelle pour pouvoir "report" et permettre
    le pruning par Optuna (ex: MedianPruner). On utilise KFold pour générer
    des scores intermédiaires.
    """
    # Optuna suggère les hyperparamètres
    n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
    max_depth = trial.suggest_int('max_depth', 5, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    # Créer une nouvelle pipeline avec ces hyperparamètres
    rf_model_trial = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    pipeline_trial = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('selector', feature_selector),
        ('rf', rf_model_trial)
    ])

    # Boucle CV manuelle pour pouvoir reporter les scores de chaque fold
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pipeline_trial.fit(X_tr, y_tr)
        preds = pipeline_trial.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        neg_rmse = -rmse  # on garde la convention 'plus grand = meilleur'
        fold_scores.append(neg_rmse)

        # Report pour le pruner d'Optuna
        trial.report(neg_rmse, step=fold_idx)
        if trial.should_prune():
            raise TrialPruned()

    # Retourner la moyenne négative du RMSE (direction maximize)
    return float(np.mean(fold_scores))


# Étape E : Lancement de l'étude Optuna
print("Lancement de l'optimisation Optuna (recherche bayésienne intelligente + pruning)...")
print("Cela va tester jusqu'à 50 combinaisons mais pourra s'arrêter plus tôt si inutile...\n")

# Sampler Bayésien (TPE = Tree-structured Parzen Estimator) et pruner
sampler = TPESampler(seed=42)
pruner = MedianPruner(n_warmup_steps=5)

# Callback pour stopper l'étude si pas d'amélioration sur N trials consécutifs
def stop_if_no_improvement_factory(patience=10):
    state = {'best_value': None, 'no_improve': 0}

    def _callback(study, trial):
        # study.best_value suit la direction (ici maximize)
        if state['best_value'] is None or study.best_value > state['best_value']:
            state['best_value'] = study.best_value
            state['no_improve'] = 0
        else:
            state['no_improve'] += 1

        if state['no_improve'] >= patience:
            print(f"Aucun progrès depuis {patience} essais — arrêt de l'étude Optuna.")
            study.stop()

    return _callback

early_stop_callback = stop_if_no_improvement_factory(patience=10)

study = optuna.create_study(
    direction='maximize',  # On maximise car on retourne neg_rmse
    sampler=sampler,
    pruner=pruner
)

# Lancement de l'optimisation avec callback d'arrêt anticipé
study.optimize(objective, n_trials=50, callbacks=[early_stop_callback], show_progress_bar=True)

print("-" * 50)
print("Optimisation terminée !")
print(f"Meilleur score CV (neg_rmse) : {study.best_value:.4f}")
print(f"Meilleurs hyperparamètres trouvés :\n{study.best_params}")
print("-" * 50)

# Réentraîner le meilleur modèle avec les meilleurs paramètres
best_params = study.best_params
rf_best = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    max_features=best_params['max_features'],
    random_state=42,
    n_jobs=-1
)

best_m3_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', feature_selector),
    ('rf', rf_best)
])

# Fit sur tout le train set
best_m3_pipeline.fit(X_train, y_train)

# Évaluation sur le Train Set
y_pred_train = best_m3_pipeline.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f"RMSE (Log) sur le Train Set : {rmse_train:.4f}")

[I 2026-06-13 08:59:28,182] A new study created in memory with name: no-name-471924b2-ce71-4942-ac55-e8687df653a2


Lancement de l'optimisation Optuna (recherche bayésienne intelligente + pruning)...
Cela va tester jusqu'à 50 combinaisons mais pourra s'arrêter plus tôt si inutile...



Best trial: 0. Best value: -0.135428:   2%|▏         | 1/50 [00:01<01:18,  1.60s/it]

[I 2026-06-13 08:59:29,785] Trial 0 finished with value: -0.13542796903628893 and parameters: {'n_estimators': 200, 'max_depth': 48, 'min_samples_split': 15, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.13542796903628893.


Best trial: 0. Best value: -0.135428:   4%|▍         | 2/50 [00:02<01:09,  1.45s/it]

...

[I 2026-06-13 08:59:49,863] Trial 10 finished with value: -0.1310088279563848 and parameters: {'n_estimators': 500, 'max_depth': 37, 'min_samples_split': 6, 'max_features': 'log2'}. Best is trial 7 with value: -0.1300613269545535.


Best trial: 11. Best value: -0.130012:  24%|██▍       | 12/50 [00:24<01:32,  2.44s/it]

[I 2026-06-13 08:59:52,825] Trial 11 finished with value: -0.1300119917226243 and parameters: {'n_estimators': 500, 'max_depth': 26, 'min_samples_split': 2, 'max_features': 'log2'}. Best is trial 11 with value: -0.1300119917226243.


Best trial: 11. Best value: -0.130012:  26%|██▌       | 13/50 [00:27<01:33,  2.53s/it]

[I 2026-06-13 08:59:55,561] Trial 12 finished with value: -0.1310831893410425 and parameters: {'n_estimators': 500, 'max_depth': 27, 'min_samples_split': 5, 'max_features': 'log2'}. Best is trial 11 with value: -0.1300119917226243.


# 6. Génération de la Soumission Kaggle
- On utilise le meilleur pipeline trouvé par Optuna pour prédire les prix du jeu de test (`test.csv`).
- Les prédictions sont en log (car le modèle a été entraîné sur `log1p(SalePrice)`), on les reconvertit en vrais dollars avec `expm1`.
- Le fichier CSV généré respecte le format attendu par Kaggle : deux colonnes `Id` et `SalePrice`.

In [ ]:
# On prédit avec le meilleur modèle trouvé
preds_log_m3 = best_m3_pipeline.predict(X_test)
preds_dollars_m3 = np.expm1(preds_log_m3) # Retour au prix en dollars

submission_m3 = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds_dollars_m3
})

fichier_soumission = 'M3_RandomForest_Submission.csv'
submission_m3.to_csv(fichier_soumission, index=False)
print(f"Fichier '{fichier_soumission}' prêt ! Envoie-le sur Kaggle pour voir notre nouveau score.")

# 7. Évaluation Finale : Les 3 Critères du Groupe
- **Mathématique (RMSE)** : Récupéré depuis l'optimisation Optuna (étape 4). C'est le RMSE moyen en validation croisée sur l'échelle logarithmique.
- **IT (Temps d'entraînement)** : Temps nécessaire pour entraîner la pipeline complète (preprocessing + sélection Lasso + Random Forest) sur 100% du train set.
- **Métier (% d'écart médian)** : Médiane des écarts relatifs entre le prix prédit et le prix réel, en pourcentage. La médiane est robuste aux valeurs extrêmes et représente l'erreur « typique » du modèle.
- On affiche également le Top 15 des variables les plus importantes selon le Random Forest, pour comprendre quels facteurs influencent le plus la prédiction du prix.

In [ ]:
import time
from sklearn.model_selection import cross_val_predict

# ── 1. MATHÉMATIQUE : RMSE (récupéré depuis Optuna, étape 4) ──
rmse_log = -study.best_value  # study.best_value est en neg_rmse

# ── 2. IT : Temps d'entraînement (un seul fit chronométré) ──
start_time = time.time()
best_m3_pipeline.fit(X_train, y_train)
train_time = time.time() - start_time

# ── 3. MÉTIER : % d'écart médian (prédictions OOF honnêtes) ──
kf_eval = KFold(n_splits=5, shuffle=True, random_state=42)
oof_pred_log = cross_val_predict(best_m3_pipeline, X_train, y_train, cv=kf_eval, n_jobs=-1)
oof_pred_price = np.expm1(oof_pred_log)
y_train_price = np.expm1(y_train)

ecart_pct = np.abs(oof_pred_price - y_train_price) / y_train_price * 100
ecart_median = np.median(ecart_pct)

# ── AFFICHAGE ──
print("=" * 50)
print("  ÉVALUATION FINALE — MODÈLE M3 RANDOM FOREST")
print("=" * 50)
print(f"  Mathématique (RMSE Log) : {rmse_log:.5f}")
print(f"  IT (Temps entraînement) : {train_time:.2f} secondes")
print(f"  Métier (Écart médian)   : {ecart_median:.2f} %")
print("=" * 50)

# ── GRAPHE : Feature Importances (Top 15) ──
feature_names = best_m3_pipeline.named_steps['preprocessor'].get_feature_names_out()
lasso_mask = best_m3_pipeline.named_steps['selector'].get_support()
selected_feature_names = feature_names[lasso_mask]
importances = best_m3_pipeline.named_steps['rf'].feature_importances_

importance_df = pd.DataFrame({
    'Variable': selected_feature_names,
    'Importance': importances
})
importance_df['Variable'] = importance_df['Variable'].str.replace(r'^(num|cat_none|cat_freq)__', '', regex=True)
importance_df = importance_df.sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Importance', y='Variable', palette='viridis')
plt.title('Top 15 des variables les plus importantes (Random Forest)')
plt.xlabel('Importance relative')
plt.ylabel('')
plt.tight_layout()
plt.show()